In [1]:
!pip install web3==6.0.0

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of eth-abi to determine which version is compatible with other requirements. This could take a while.
  Using cached eth_abi-6.0.0b1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/569.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/569.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/569.0 kB ? eta -:--:--
   ------------------ --------------------- 262.1/569.0 kB ? eta -:--:--
   ------------------ --------------------- 262.1/569.0 kB ? eta -:--:--
   --------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
from dotenv import load_dotenv
from web3 import Web3, HTTPProvider

load_dotenv()
link = f"https://sepolia.infura.io/v3/{os.environ.get('INFURA_TOKEN')}"
w3 = Web3(HTTPProvider(link))
print("Connection is established:", w3.is_connected())

Connection is established: True


In [11]:
CONTRACT_ADDRESS = Web3.to_checksum_address('0x6ec02781E3a7c9D43D5926c80ECdE45397dA355e')

CONTRACT_ABI = [
    {
        "inputs": [{"name": "account", "type": "address"}],
        "name": "solved",
        "outputs": [{"name": "", "type": "bool"}],
        "stateMutability": "view",
        "type": "function",
    },
    {
        "inputs": [{"name": "account", "type": "address"}],
        "name": "addressName",
        "outputs": [{"name": "", "type": "string"}],
        "stateMutability": "view",
        "type": "function",
    },
    {
        "inputs": [{"name": "account", "type": "address"}],
        "name": "addressId",
        "outputs": [{"name": "", "type": "uint256"}],
        "stateMutability": "view",
        "type": "function",
    },
    {
        "inputs": [
            {"name": "studentId", "type": "uint256"},
            {"name": "nameSurname", "type": "string"},
            {"name": "answer", "type": "bytes32"},
        ],
        "name": "solve",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function",
    },
]

contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=CONTRACT_ABI)
print("Contract instance created at:", CONTRACT_ADDRESS)

Contract instance created at: 0x6ec02781E3a7c9D43D5926c80ECdE45397dA355e


In [12]:
TX_HASH = '0x7419742552e9b84933c9951a6240f34df851ff882968adfbca1ed6248cda1c27'

tx = w3.eth.get_transaction(TX_HASH)
sender = tx['from']

is_solved = contract.functions.solved(sender).call()
name = contract.functions.addressName(sender).call()
student_id = contract.functions.addressId(sender).call()

print(f"Sender address : {sender}")
print(f"solved         : {is_solved}")
print(f"addressName    : {name!r}")
print(f"addressId      : {student_id}")

Sender address : 0x02d909fBDE0Aeafb3Ba9848C1D7b387577f26131
solved         : True
addressName    : 'shahdSherif'
addressId      : 38151


In [13]:
receipt = w3.eth.get_transaction_receipt(TX_HASH)
block = w3.eth.get_block(tx['blockNumber'])

fn_name, fn_args = contract.decode_function_input(tx['input'])

print("=== Transaction ===")
print(f"Hash        : {TX_HASH}")
print(f"Status      : {'success' if receipt['status'] == 1 else 'reverted'}")
print(f"From        : {tx['from']}")
print(f"To          : {tx['to']}")
print(f"Block       : {tx['blockNumber']}")
print(f"Nonce       : {tx['nonce']}")
print(f"Gas limit   : {tx['gas']}")
print(f"Gas used    : {receipt['gasUsed']}")
print(f"Gas price   : {Web3.from_wei(tx['gasPrice'], 'gwei'):.4f} gwei")
print(f"Value       : {Web3.from_wei(tx['value'], 'ether')} ETH")
print()
print("=== Decoded call ===")
print(f"Function    : {fn_name.fn_name}")
for arg_name, arg_val in fn_args.items():
    display_val = arg_val.hex() if isinstance(arg_val, bytes) else arg_val
    print(f"  {arg_name:<12}: {display_val}")
print()
print("=== Block ===")
print(f"Timestamp   : {block['timestamp']}")
print(f"Parent hash : {block['parentHash'].hex()}")
print(f"Tx count    : {len(block['transactions'])}")

=== Transaction ===
Hash        : 0x7419742552e9b84933c9951a6240f34df851ff882968adfbca1ed6248cda1c27
Status      : success
From        : 0x02d909fBDE0Aeafb3Ba9848C1D7b387577f26131
To          : 0x6ec02781E3a7c9D43D5926c80ECdE45397dA355e
Block       : 10628981
Nonce       : 4
Gas limit   : 92368
Gas used    : 91422
Gas price   : 5.5298 gwei
Value       : 0 ETH

=== Decoded call ===
Function    : solve
  studentId   : 38151
  nameSurname : shahdSherif
  answer      : 5d53577dae6ac01976df0ce05a909211c506204d9819b1dafe4ffbeecedbe298

=== Block ===
Timestamp   : 1775810364
Parent hash : cc541eb67de6cd4b82b3b95db28c3ab8f6dd98a1df0edf4e0858fef2d1dcedbd
Tx count    : 165


In [14]:
# Add WALLET_PRIVATE_KEY=0x<your_key> to .env before running this cell.
# To generate a fresh agent wallet instead, run:
#   acct = w3.eth.account.create()
#   print("Address   :", acct.address)
#   print("Priv key  :", acct.key.hex())
# Save the printed private key to .env as WALLET_PRIVATE_KEY, then fund it
# from a Sepolia faucet (e.g. https://faucets.chain.link/sepolia).

private_key = os.environ.get('WALLET_PRIVATE_KEY')
if private_key is None:
    raise EnvironmentError("WALLET_PRIVATE_KEY not set in .env")

wallet = w3.eth.account.from_key(private_key)
balance_wei = w3.eth.get_balance(wallet.address)

print(f"Wallet address : {wallet.address}")
print(f"Balance        : {Web3.from_wei(balance_wei, 'ether'):.6f} ETH")

Wallet address : 0x02d909fBDE0Aeafb3Ba9848C1D7b387577f26131
Balance        : 0.590593 ETH


In [15]:
SEPOLIA_CHAIN_ID = 11155111

# Replace with test values different from 38151 / "shahdSherif"
test_student_id = 99999
test_name_surname = "testAgent"

answer = w3.solidity_keccak(["uint256", "string"], [test_student_id, test_name_surname])
print(f"Computed answer: {answer.hex()}")

tx_data = contract.functions.solve(
    test_student_id,
    test_name_surname,
    answer,
).build_transaction({
    'from': wallet.address,
    'nonce': w3.eth.get_transaction_count(wallet.address),
    'chainId': SEPOLIA_CHAIN_ID,
})

signed_tx = w3.eth.account.sign_transaction(tx_data, private_key=private_key)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
print(f"Sent tx: {tx_hash.hex()}")

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print(f"Status : {'success' if receipt['status'] == 1 else 'reverted'}")
print(f"Gas used: {receipt['gasUsed']}")

# Verify on-chain state updated
print(f"solved[{wallet.address}] = {contract.functions.solved(wallet.address).call()}")

Computed answer: 7b3e42d77c28772202849aacbf689fa9da48091d849c30ac2aeba9955aa03a38
Sent tx: 285143d7ca67427f5cbec59d7ea1f49c2306c39279a907223460b3e8a4f5dc84
Status : success
Gas used: 37310
solved[0x02d909fBDE0Aeafb3Ba9848C1D7b387577f26131] = True


In [ ]:
#1) Read directly from EVM contract storage
contractAddress = '0x6ec02781E3a7c9D43D5926c80ECdE45397dA355e'#'0x9e2A277fBf45873f011A91dc371F504fC168a116'
print("This is the data: ", w3.to_int(w3.eth.get_storage_at(contractAddress, 0)));
print("This is the data: ", w3.to_int(w3.eth.get_storage_at(contractAddress, 1)));
print("This is the data: ", w3.to_int(w3.eth.get_storage_at(contractAddress, 2)));
print("This is the data: ", w3.to_int(w3.eth.get_storage_at(contractAddress, 3)));
print("This is the data: ", w3.to_int(w3.eth.get_storage_at(contractAddress, 4)));
print()

#2) An alternative way - compute the value as in contract
#from web3.middleware import geth_poa_middleware
#w3.middleware_onion.inject(geth_poa_middleware, layer=0) #this is only required for rinkleby (or mumbai etc.) we do not need a poa middleware for eth
tx = w3.eth.get_transaction('0x3464f6b26b677e3c84f50b390c66b0ef0d5411a7d0bd17b8c85870c8938c71e7')
blockNumber = tx.blockNumber;
print('Transaction is in block ', blockNumber)
block = w3.eth.get_block(blockNumber);
timestamp = block.timestamp;
parentHash = block.parentHash;
print('Timestamp: ', timestamp);
hash = w3.solidity_keccak(['bytes32', 'uint256'], [parentHash, timestamp])
print('Hash value is: ', hash.hex())

In [ ]:
answer = w3.solidity_keccak(
    ["uint256", "string"],
    [38151, "shahdSherif"]
)

print(answer.hex())